# Distributed Transient Analysis Validation

This notebook validates the distributed transient IR-drop analysis using the `netlist_sampled` test case
with the **distributed DDM solver** (`DistributedDDMSolver`).

**Key differences from the flat transient notebook:**
- Uses pre-parsed tile pkl files instead of `NetlistParser`
- Single `DistributedDDMSolver` handles DC, quasi-static, and transient modes
- Current source data lives on tile workers (preprocessed via `preprocess_sources()`)
- Peak tracking is lazy -- stays on workers until `.as_flat()` is called
- Requires `model.shutdown()` at the end

**Validation points:**
1. Distributed DC solve produces a reference IR-drop
2. Quasi-static and transient (BE/Trap) analyses run correctly
3. Capacitive smoothing effects are visible in transient vs quasi-static comparison
4. Time-domain waveforms and peak tracking work end-to-end

## 1. Setup and Imports

In [1]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import interpolate

warnings.filterwarnings('ignore')

# Change to project root so TileConfig relative paths resolve correctly
os.chdir(Path(__file__).parent.parent if '__file__' in dir() else Path('..'))

from distributed.model import create_distributed_model, load_distributed_partitions
from distributed.solver import DistributedDDMSolver

%matplotlib inline
print(f"Working directory: {os.getcwd()}")
print("Imports successful!")

Working directory: /wv/bwdev1/patrasej/dev/sigma_dvd/prototype
Imports successful!


## 2. Load Distributed Model

In [2]:
pkl_dir = Path('netlist/netlist_sampled/distributed_pkl')

print(f"Loading distributed partitions from {pkl_dir}...")
bundle = load_distributed_partitions(str(pkl_dir))

model = create_distributed_model(bundle, backend='ray')

print(f"\nModel created successfully!")
print(f"  Vdd: {model.vdd} V")
print(f"  Net: {model.net_name}")
print(f"  Tiles: {len(model.metadata.tile_configs)}")
print(f"  Pad nodes: {model.pad_nodes}")

print(f"\nPer-tile sizes:")
for tid, n_int in sorted(model.tile_interior_counts.items()):
    n_bnd = len(model.tile_boundary_nodes[tid])
    print(f"  Tile {tid}: {n_int} interior, {n_bnd} boundary")

Loading distributed partitions from netlist/netlist_sampled/distributed_pkl...


2026-03-14 14:23:14,217	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-14 14:23:23,965	INFO worker.py:2013 -- Started a local Ray instance.



Model created successfully!
  Vdd: 0.66 V
  Net: VDD_XLV
  Tiles: 9
  Pad nodes: {'VDD_XLV_vsrc'}

Per-tile sizes:
  Tile (0, 0): 15189 interior, 471 boundary
  Tile (0, 1): 15049 interior, 508 boundary
  Tile (0, 2): 15118 interior, 481 boundary
  Tile (1, 0): 15171 interior, 622 boundary
  Tile (1, 1): 15026 interior, 658 boundary
  Tile (1, 2): 15098 interior, 630 boundary
  Tile (2, 0): 13453 interior, 178 boundary
  Tile (2, 1): 13864 interior, 217 boundary
  Tile (2, 2): 15616 interior, 192 boundary


## 3. Static DC IR-Drop (Reference)

In [3]:
solver = DistributedDDMSolver(model)

t0 = time.perf_counter()
dc_result = solver.solve_dc(verbose=True)
dc_time = time.perf_counter() - t0

ir_drop = dc_result.ir_drop
max_drop_node = max(ir_drop, key=ir_drop.get)
max_drop = ir_drop[max_drop_node]

print(f"\nDC Analysis Results (time: {dc_time*1000:.1f} ms):")
print(f"  Max IR-drop: {max_drop*1000:.4f} mV")
print(f"  Max IR-drop node: {max_drop_node}")
print(f"  Min voltage: {model.vdd - max_drop:.6f} V")
print(f"  Mean IR-drop: {np.mean(list(ir_drop.values()))*1000:.4f} mV")

Matrix factorization (cholmod(mode=auto, ordering=default, idx=auto)): CSC conversion 8.24 ms, factorization 2126.01 ms

DC Analysis Results (time: 5041.3 ms):
  Max IR-drop: 1.6353 mV
  Max IR-drop node: 1673000_1958400_M0
  Min voltage: 0.658365 V
  Mean IR-drop: 0.5094 mV


## 4. Preprocess Current Sources

Load and optionally smooth time-varying current sources on tile workers.

In [4]:
# Simulation parameters
t_start = 0.0
t_end = 10e-9     # 10 ns
dt = 10e-12        # 10 ps timestep
n_points = 1001    # for quasi-static

print(f"Preprocessing sources (dt={dt*1e12:.0f} ps, t_end={t_end*1e9:.0f} ns)...")
t0 = time.perf_counter()
smoothed = solver.preprocess_sources(
    time_step=dt,
    t_start=t_start,
    t_end=t_end,
    smooth=True,
    verbose=True,
)
preprocess_time = time.perf_counter() - t0

print(f"\nPreprocessing time: {preprocess_time*1000:.1f} ms")
print(f"Smoothed: {smoothed.smoothed}")
print(f"\nPer-tile source stats:")
total_sources = 0
for tid, stats in sorted(smoothed.per_tile_stats.items()):
    n = stats.get('n_sources', 0)
    total_sources += n
    print(f"  Tile {tid}: {n} sources, "
          f"{stats.get('n_pulses', 0)} pulses, "
          f"{stats.get('n_pwls', 0)} PWLs")
print(f"  Total: {total_sources} sources")

Preprocessing sources (dt=10 ps, t_end=10 ns)...

Preprocessing time: 4712.9 ms
Smoothed: True

Per-tile source stats:
  Tile (0, 0): 7538 sources, 11836 pulses, 0 PWLs
  Tile (0, 1): 7834 sources, 16056 pulses, 0 PWLs
  Tile (0, 2): 7000 sources, 5281 pulses, 0 PWLs
  Tile (1, 0): 6375 sources, 6754 pulses, 0 PWLs
  Tile (1, 1): 7525 sources, 8797 pulses, 0 PWLs
  Tile (1, 2): 7094 sources, 9659 pulses, 0 PWLs
  Tile (2, 0): 3898 sources, 5014 pulses, 0 PWLs
  Tile (2, 1): 5059 sources, 1486 pulses, 0 PWLs
  Tile (2, 2): 6826 sources, 7732 pulses, 0 PWLs
  Total: 59149 sources


## 5. Select Tracking Nodes

Pick the worst IR-drop nodes from the DC solve to track full waveforms.

In [5]:
# Select top-5 worst nodes from DC result for waveform tracking
sorted_by_drop = sorted(ir_drop.items(), key=lambda x: x[1], reverse=True)
track_nodes = [n for n, _ in sorted_by_drop[:5]]

print("Tracking waveforms for top-5 worst DC nodes:")
for n, d in sorted_by_drop[:5]:
    print(f"  {n}: {d*1000:.4f} mV")

Tracking waveforms for top-5 worst DC nodes:
  1673000_1958400_M0: 1.6353 mV
  1638800_1958400_M0: 1.5934 mV
  989000_1588800_M0: 1.5670 mV
  1663000_1958400_M0: 1.5573 mV
  1653190_1958400_M0: 1.4679 mV


## 6. Quasi-Static Analysis

Batch DC solves at discrete time points (no capacitance effects).

In [6]:
print(f"Running quasi-static analysis ({n_points} time points)...")
t0 = time.perf_counter()
qs_result = solver.solve_quasi_static(
    t_start=t_start,
    t_end=t_end,
    n_points=n_points,
    smoothed_sources=smoothed,
    track_nodes=track_nodes,
    verbose=True,
)
qs_time = time.perf_counter() - t0

print(f"\nQuasi-Static Results (time: {qs_time:.1f} s):")
print(f"  Peak IR-drop: {qs_result.peak_ir_drop*1000:.4f} mV")
print(f"  Peak time: {qs_result.peak_ir_drop_time*1e9:.2f} ns")
print(f"  Time points: {len(qs_result.t_array)}")

Running quasi-static analysis (1001 time points)...
Matrix factorization (cholmod(mode=auto, ordering=default, idx=auto)): CSC conversion 7.10 ms, factorization 1990.71 ms

Quasi-Static Results (time: 52.9 s):
  Peak IR-drop: 62.5698 mV
  Peak time: 2.28 ns
  Time points: 1001


## 7. Transient Analysis (Backward Euler)

RC transient analysis with implicit Backward Euler time integration.

In [7]:
print(f"Running transient analysis (BE, dt={dt*1e12:.0f} ps)...")
t0 = time.perf_counter()
be_result = solver.solve_transient(
    t_start=t_start,
    t_end=t_end,
    dt=dt,
    method='be',
    smoothed_sources=smoothed,
    track_nodes=track_nodes,
    verbose=True,
)
be_time = time.perf_counter() - t0

print(f"\nTransient (BE) Results (time: {be_time:.1f} s):")
print(f"  Peak IR-drop: {be_result.peak_ir_drop*1000:.4f} mV")
print(f"  Peak time: {be_result.peak_ir_drop_time*1e9:.2f} ns")
print(f"  Has capacitance: {be_result.has_capacitance}")
print(f"  Integration method: {be_result.integration_method}")

Running transient analysis (BE, dt=10 ps)...
Matrix factorization (cholmod(mode=auto, ordering=default, idx=auto)): CSC conversion 6.34 ms, factorization 2088.73 ms
Matrix factorization (cholmod(mode=auto, ordering=default, idx=auto)): CSC conversion 5.99 ms, factorization 2304.43 ms

Transient (BE) Results (time: 59.8 s):
  Peak IR-drop: 50.9123 mV
  Peak time: 4.78 ns
  Has capacitance: True
  Integration method: be


## 8. Transient Analysis (Trapezoidal)

RC transient analysis with second-order Trapezoidal integration.

In [ ]:
print(f"Running transient analysis (Trap, dt={dt*1e12:.0f} ps)...")
t0 = time.perf_counter()
trap_result = solver.solve_transient(
    t_start=t_start,
    t_end=t_end,
    dt=dt,
    method='trap',
    smoothed_sources=smoothed,
    track_nodes=track_nodes,
    verbose=True,
)
trap_time = time.perf_counter() - t0

print(f"\nTransient (Trap) Results (time: {trap_time:.1f} s):")
print(f"  Peak IR-drop: {trap_result.peak_ir_drop*1000:.4f} mV")
print(f"  Peak time: {trap_result.peak_ir_drop_time*1e9:.2f} ns")
print(f"  Has capacitance: {trap_result.has_capacitance}")
print(f"  Integration method: {trap_result.integration_method}")

## 9. Validation: Compare Methods

In [ ]:
print("=== Method Comparison ===")
print(f"\n{'Method':<20} {'Peak IR-drop (mV)':>18} {'Peak Time (ns)':>15}")
print("-" * 55)
print(f"{'Quasi-Static':<20} {qs_result.peak_ir_drop*1000:>18.4f} {qs_result.peak_ir_drop_time*1e9:>15.2f}")
print(f"{'Transient (BE)':<20} {be_result.peak_ir_drop*1000:>18.4f} {be_result.peak_ir_drop_time*1e9:>15.2f}")
print(f"{'Transient (Trap)':<20} {trap_result.peak_ir_drop*1000:>18.4f} {trap_result.peak_ir_drop_time*1e9:>15.2f}")

# Compute differences
diff_be = abs(be_result.peak_ir_drop - qs_result.peak_ir_drop)
diff_trap = abs(trap_result.peak_ir_drop - qs_result.peak_ir_drop)
diff_be_trap = abs(be_result.peak_ir_drop - trap_result.peak_ir_drop)

print(f"\nPeak Differences:")
print(f"  QS vs BE:     {diff_be*1e6:.2f} uV")
print(f"  QS vs Trap:   {diff_trap*1e6:.2f} uV")
print(f"  BE vs Trap:   {diff_be_trap*1e6:.2f} uV")

print(f"\nNote: With capacitance, Backward Euler is more damped (lower peaks),")
print(f"while Trapezoidal is second-order accurate. Without capacitance,")
print(f"all methods should match quasi-static exactly.")

## 10. Visualization: Time Series Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Max IR-drop vs time
ax = axes[0, 0]
ax.plot(qs_result.t_array * 1e9, qs_result.max_ir_drop_per_time * 1000,
        'b-', linewidth=2, label='Quasi-Static')
ax.plot(be_result.t_array * 1e9, be_result.max_ir_drop_per_time * 1000,
        'r--', linewidth=1.5, label='Transient (BE)')
ax.plot(trap_result.t_array * 1e9, trap_result.max_ir_drop_per_time * 1000,
        'g:', linewidth=1.5, label='Transient (Trap)')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Max IR-Drop (mV)')
ax.set_title('Maximum IR-Drop vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Top-right: Total current vs time
ax = axes[0, 1]
ax.plot(qs_result.t_array * 1e9, qs_result.total_current_per_time,
        'b-', linewidth=2, label='QS Total Current')
ax.plot(be_result.t_array * 1e9, be_result.total_current_per_time,
        'r--', linewidth=1.5, label='BE Total Current')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Current (mA)')
ax.set_title('Total Load Current vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

# Bottom-left: Tracked node waveforms (quasi-static)
ax = axes[1, 0]
for node in track_nodes[:5]:
    if node in qs_result.tracked_ir_drop:
        waveform = qs_result.tracked_ir_drop[node]
        label = f'{str(node)[:20]}...' if len(str(node)) > 20 else str(node)
        ax.plot(qs_result.t_array * 1e9, waveform * 1000,
                linewidth=1.5, label=label)
ax.set_xlabel('Time (ns)')
ax.set_ylabel('IR-Drop (mV)')
ax.set_title('IR-Drop Waveforms (Worst Nodes, QS)')
ax.legend(fontsize='small')
ax.grid(True, alpha=0.3)

# Bottom-right: Method difference over time
ax = axes[1, 1]
# Interpolate transient results to QS time grid for comparison
be_interp = interpolate.interp1d(be_result.t_array,
                                  be_result.max_ir_drop_per_time,
                                  fill_value='extrapolate')
trap_interp = interpolate.interp1d(trap_result.t_array,
                                    trap_result.max_ir_drop_per_time,
                                    fill_value='extrapolate')

# Use the overlapping time range
t_common = qs_result.t_array[
    (qs_result.t_array >= be_result.t_array[0]) &
    (qs_result.t_array <= be_result.t_array[-1])
]
if len(t_common) > 0:
    qs_at_common = interpolate.interp1d(
        qs_result.t_array, qs_result.max_ir_drop_per_time,
        fill_value='extrapolate'
    )(t_common)
    diff_be_arr = np.abs(qs_at_common - be_interp(t_common)) * 1e6  # uV
    diff_trap_arr = np.abs(qs_at_common - trap_interp(t_common)) * 1e6

    ax.semilogy(t_common * 1e9, diff_be_arr + 1e-6,
                'r-', linewidth=1.5, label='QS vs BE')
    ax.semilogy(t_common * 1e9, diff_trap_arr + 1e-6,
                'g--', linewidth=1.5, label='QS vs Trap')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Difference (uV)')
ax.set_title('Max IR-Drop Difference Between Methods')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Visualization: Per-Node Waveform Comparison

Compare QS vs BE vs Trap waveforms for the worst node.

In [ ]:
# Find a node tracked in all three results
common_tracked = (
    set(qs_result.tracked_ir_drop.keys()) &
    set(be_result.tracked_ir_drop.keys()) &
    set(trap_result.tracked_ir_drop.keys())
)

if common_tracked:
    test_node = max(common_tracked,
                    key=lambda n: np.max(qs_result.tracked_ir_drop[n]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Overlaid waveforms
    ax = axes[0]
    ax.plot(qs_result.t_array * 1e9,
            qs_result.tracked_ir_drop[test_node] * 1000,
            'b-', linewidth=2, label='Quasi-Static')
    ax.plot(be_result.t_array * 1e9,
            be_result.tracked_ir_drop[test_node] * 1000,
            'r--', linewidth=1.5, label='Transient (BE)')
    ax.plot(trap_result.t_array * 1e9,
            trap_result.tracked_ir_drop[test_node] * 1000,
            'g:', linewidth=1.5, label='Transient (Trap)')
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('IR-Drop (mV)')
    ax.set_title(f'Node {test_node}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: Difference from QS
    ax = axes[1]
    be_node_interp = interpolate.interp1d(
        be_result.t_array, be_result.tracked_ir_drop[test_node],
        fill_value='extrapolate'
    )
    trap_node_interp = interpolate.interp1d(
        trap_result.t_array, trap_result.tracked_ir_drop[test_node],
        fill_value='extrapolate'
    )

    if len(t_common) > 0:
        qs_node_at_common = interpolate.interp1d(
            qs_result.t_array, qs_result.tracked_ir_drop[test_node],
            fill_value='extrapolate'
        )(t_common)
        ax.plot(t_common * 1e9,
                (qs_node_at_common - be_node_interp(t_common)) * 1e6,
                'r-', linewidth=1.5, label='QS - BE')
        ax.plot(t_common * 1e9,
                (qs_node_at_common - trap_node_interp(t_common)) * 1e6,
                'g--', linewidth=1.5, label='QS - Trap')
    ax.set_xlabel('Time (ns)')
    ax.set_ylabel('Difference (uV)')
    ax.set_title(f'Waveform Difference at {test_node}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Quantitative comparison
    if len(t_common) > 0:
        diff_be_node = np.abs(qs_node_at_common - be_node_interp(t_common))
        diff_trap_node = np.abs(qs_node_at_common - trap_node_interp(t_common))
        print(f"Node: {test_node}")
        print(f"  Max diff QS vs BE:   {np.max(diff_be_node)*1e6:.2f} uV")
        print(f"  Max diff QS vs Trap: {np.max(diff_trap_node)*1e6:.2f} uV")
        print(f"  RMS diff QS vs BE:   {np.sqrt(np.mean(diff_be_node**2))*1e6:.2f} uV")
        print(f"  RMS diff QS vs Trap: {np.sqrt(np.mean(diff_trap_node**2))*1e6:.2f} uV")
else:
    print("No common tracked nodes found across all three results.")

## 12. Peak IR-Drop Distribution

Collect per-node peak data from workers and compare distributions.

In [ ]:
# Collect peak data from workers (lazy -- first call triggers collection)
print("Collecting peak data from workers...")
t0 = time.perf_counter()
qs_peaks = qs_result.as_flat()
be_peaks = be_result.as_flat()
trap_peaks = trap_result.as_flat()
collect_time = time.perf_counter() - t0
print(f"Collection time: {collect_time*1000:.1f} ms")
print(f"Nodes with peaks: QS={len(qs_peaks)}, BE={len(be_peaks)}, Trap={len(trap_peaks)}")

# Extract peak drop values
qs_drops = np.array([d for d, _ in qs_peaks.values()]) * 1000  # mV
be_drops = np.array([d for d, _ in be_peaks.values()]) * 1000
trap_drops = np.array([d for d, _ in trap_peaks.values()]) * 1000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram of peak IR-drops
ax = axes[0]
bins = np.linspace(0, max(qs_drops.max(), be_drops.max(), trap_drops.max()) * 1.05, 50)
ax.hist(qs_drops, bins=bins, alpha=0.5, label='Quasi-Static', color='blue')
ax.hist(be_drops, bins=bins, alpha=0.5, label='Transient (BE)', color='red')
ax.hist(trap_drops, bins=bins, alpha=0.5, label='Transient (Trap)', color='green')
ax.set_xlabel('Peak IR-Drop (mV)')
ax.set_ylabel('Node Count')
ax.set_title('Peak IR-Drop Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: CDF comparison
ax = axes[1]
for drops, label, color in [
    (qs_drops, 'Quasi-Static', 'blue'),
    (be_drops, 'Transient (BE)', 'red'),
    (trap_drops, 'Transient (Trap)', 'green'),
]:
    sorted_drops = np.sort(drops)
    cdf = np.arange(1, len(sorted_drops) + 1) / len(sorted_drops)
    ax.plot(sorted_drops, cdf, linewidth=1.5, label=label, color=color)
ax.set_xlabel('Peak IR-Drop (mV)')
ax.set_ylabel('CDF')
ax.set_title('Cumulative Distribution of Peak IR-Drop')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\n{'Statistic':<20} {'QS (mV)':>10} {'BE (mV)':>10} {'Trap (mV)':>10}")
print("-" * 52)
print(f"{'Max'::<20} {qs_drops.max():>10.4f} {be_drops.max():>10.4f} {trap_drops.max():>10.4f}")
print(f"{'Mean'::<20} {qs_drops.mean():>10.4f} {be_drops.mean():>10.4f} {trap_drops.mean():>10.4f}")
print(f"{'P99'::<20} {np.percentile(qs_drops, 99):>10.4f} {np.percentile(be_drops, 99):>10.4f} {np.percentile(trap_drops, 99):>10.4f}")
print(f"{'P95'::<20} {np.percentile(qs_drops, 95):>10.4f} {np.percentile(be_drops, 95):>10.4f} {np.percentile(trap_drops, 95):>10.4f}")

## 13. Performance Summary

In [ ]:
print("=== Performance Summary ===")
print(f"\nTiles: {len(model.metadata.tile_configs)}")
print(f"Backend: {type(model.backend).__name__}")

print(f"\n{'Phase':<25} {'Wall Time':>12}")
print("-" * 39)
print(f"{'Preprocess sources':<25} {preprocess_time*1000:>10.1f} ms")
print(f"{'DC solve':<25} {dc_time*1000:>10.1f} ms")
print(f"{'Quasi-static':<25} {qs_time:>10.1f} s")
print(f"{'Transient (BE)':<25} {be_time:>10.1f} s")
print(f"{'Transient (Trap)':<25} {trap_time:>10.1f} s")

n_qs_steps = len(qs_result.t_array)
n_be_steps = len(be_result.t_array)
n_trap_steps = len(trap_result.t_array)

print(f"\n{'Method':<25} {'Steps':>8} {'ms/step':>10}")
print("-" * 45)
print(f"{'Quasi-static':<25} {n_qs_steps:>8} {qs_time/n_qs_steps*1000:>10.2f}")
print(f"{'Transient (BE)':<25} {n_be_steps:>8} {be_time/n_be_steps*1000:>10.2f}")
print(f"{'Transient (Trap)':<25} {n_trap_steps:>8} {trap_time/n_trap_steps*1000:>10.2f}")

# Detailed timing breakdown from solve_metadata
print(f"\n--- QS Timing Breakdown ---")
for k, v in sorted(qs_result.solve_metadata.get('timings', {}).items()):
    print(f"  {k:<30} {v:>8.3f} s")

print(f"\n--- BE Timing Breakdown ---")
for k, v in sorted(be_result.solve_metadata.get('timings', {}).items()):
    print(f"  {k:<30} {v:>8.3f} s")

## 14. Cleanup

In [ ]:
model.shutdown()
print("Model shut down successfully.")

## 15. Conclusions

### Validation Summary

This notebook validates the **distributed** transient analysis implementation:

1. **DC reference**: The distributed DDM solver produces a baseline static IR-drop

2. **Quasi-static analysis**: Batch DC solves at discrete time points capture time-varying current behavior without capacitive effects

3. **Transient analysis (BE/Trap)**: RC time-stepping with implicit methods captures capacitive smoothing effects:
   - **Backward Euler**: First-order, numerically damped (lower peaks)
   - **Trapezoidal**: Second-order, more accurate but may show ringing

4. **Peak tracking**: Lazy collection from workers via `.as_flat()` works correctly

5. **Waveform tracking**: Per-node time-domain waveforms are recorded on workers and collected after the time loop

### Expected Behavior

- With **no capacitance**: All three methods should produce identical results
- With **capacitance**: Transient methods show smoother responses than quasi-static (capacitors filter high-frequency current variations)
- BE peak < Trap peak <= QS peak (due to numerical damping in BE)

In [ ]:
print("Notebook complete!")